# InsPLAD LVLM Benchmark — Qwen3-VL Colab smoke test

이 노트북은 **Google Colab GPU**에서 `chatgpt/insplad-benchmark-v1` 브랜치의 20-sample QA를 실제 Qwen3-VL로 돌리는 첫 baseline용입니다.

실행 순서:

1. Colab에서 **Runtime → Change runtime type → GPU** 선택
2. 아래 셀을 위에서부터 순서대로 실행
3. Google Drive에 있는 두 ZIP을 자동 탐색
4. GPU VRAM에 따라 Qwen3-VL 8B/4B 자동 선택
5. Zero-shot 20 / One-shot 20 / Structured 20 실행
6. prediction JSONL + 기존 `evaluate.py` 결과 저장
7. 마지막 셀에서 결과 전체를 ZIP으로 묶음

> 이 단계는 smoke test입니다. 전체 6,481 / 6,417 inference는 이 결과를 확인한 뒤 진행합니다.


In [ ]:
# 1) Colab 환경 확인 + 저장소/패키지 준비
import os
import subprocess
import sys
from pathlib import Path

if not Path('/content').exists():
    raise RuntimeError('이 노트북은 Google Colab 실행을 기준으로 작성되었습니다.')

REPO_URL = 'https://github.com/arnold6444/lvlm_insplad.git'
BRANCH = 'chatgpt/insplad-benchmark-v1'
REPO_DIR = Path('/content/lvlm_insplad')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.14.1', 'accelerate>=1.0', 'pillow', 'pandas', 'matplotlib', 'tqdm'
], check=True)

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[dev]'], check=True)
os.chdir(REPO_DIR)

# benchmark 코드가 깨져 있지 않은지 먼저 확인
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
print('repo ready:', REPO_DIR)


In [ ]:
# 2) Google Drive 연결 + 두 원본 ZIP 찾기
from google.colab import drive
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')

UNSUP_NAME = 'unsupervised_anomaly_detection.zip'
SUP_NAME = 'supervised_fault_classification.zip'
COPY_ZIPS_TO_LOCAL = True  # Drive에서 반복 읽는 것보다 /content로 복사하는 편이 빠름

def find_drive_file(filename: str) -> Path:
    direct = DRIVE_ROOT / filename
    if direct.exists():
        return direct
    matches = list(DRIVE_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(f'Google Drive에서 {filename!r}를 찾지 못했습니다.')
    if len(matches) > 1:
        print(f'동일 이름 파일 {len(matches)}개 발견. 첫 번째 사용:', matches[0])
    return matches[0]

unsup_drive = find_drive_file(UNSUP_NAME)
sup_drive = find_drive_file(SUP_NAME)
print('unsupervised ZIP:', unsup_drive)
print('supervised ZIP  :', sup_drive)

if COPY_ZIPS_TO_LOCAL:
    local_data = Path('/content/insplad_zips')
    local_data.mkdir(parents=True, exist_ok=True)
    unsup_zip = local_data / UNSUP_NAME
    sup_zip = local_data / SUP_NAME
    if not unsup_zip.exists() or unsup_zip.stat().st_size != unsup_drive.stat().st_size:
        shutil.copy2(unsup_drive, unsup_zip)
    if not sup_zip.exists() or sup_zip.stat().st_size != sup_drive.stat().st_size:
        shutil.copy2(sup_drive, sup_zip)
else:
    unsup_zip, sup_zip = unsup_drive, sup_drive

print('using:', unsup_zip)
print('using:', sup_zip)


In [ ]:
# 3) GPU 확인 + 모델 자동 선택
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU가 없습니다. Colab Runtime에서 GPU를 선택하세요.')

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
print('GPU :', torch.cuda.get_device_name(0))
print(f'VRAM: {vram_gb:.1f} GB')
print('BF16 supported:', torch.cuda.is_bf16_supported())

# 필요하면 직접 모델을 지정할 수 있음.
# 예: MODEL_ID_OVERRIDE = 'Qwen/Qwen3-VL-4B-Instruct'
MODEL_ID_OVERRIDE = None

if MODEL_ID_OVERRIDE:
    MODEL_ID = MODEL_ID_OVERRIDE
elif vram_gb >= 22:
    MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'
else:
    MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'

MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('MODEL_ID:', MODEL_ID)
print('dtype   :', MODEL_DTYPE)


In [ ]:
# 4) Qwen3-VL 로드
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=MODEL_DTYPE,
    device_map='auto',
    attn_implementation='sdpa',
    low_cpu_mem_usage=True,
)
model.eval()
print('model loaded')


## 데이터 읽기 방식

QA JSONL에는 ZIP 내부의 상대 경로가 저장되어 있습니다. 노트북은 원본 ZIP 전체를 풀지 않고, **현재 inference에 필요한 이미지만 `/content/insplad_image_cache`로 임시 추출**합니다.


In [ ]:
# 5) QA / ZIP 이미지 helper
import hashlib
import json
from io import BytesIO
from zipfile import ZipFile
from PIL import Image

UNSUP = ZipFile(unsup_zip)
SUP = ZipFile(sup_zip)
UNSUP_NAMES = set(UNSUP.namelist())
SUP_NAMES = set(SUP.namelist())
IMAGE_CACHE = Path('/content/insplad_image_cache')
IMAGE_CACHE.mkdir(parents=True, exist_ok=True)

ZERO_QA = REPO_DIR / 'samples/anomaly_zero_shot_20.jsonl'
ONE_QA = REPO_DIR / 'samples/anomaly_one_shot_20.jsonl'
STRUCT_QA = REPO_DIR / 'samples/defect_20.jsonl'

def read_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def resolve_member(zf: ZipFile, names: set[str], relative_path: str) -> str:
    normalized = relative_path.replace('\\', '/')
    if normalized in names:
        return normalized
    suffix = '/' + normalized
    matches = [name for name in names if name.endswith(suffix)]
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise KeyError(f'ZIP member not found: {relative_path}')
    raise KeyError(f'Ambiguous ZIP member: {relative_path} -> {len(matches)} matches')

def image_from_zip(zf: ZipFile, names: set[str], relative_path: str) -> Image.Image:
    member = resolve_member(zf, names, relative_path)
    with zf.open(member) as f:
        return Image.open(BytesIO(f.read())).convert('RGB')

def extract_image(zf: ZipFile, names: set[str], relative_path: str) -> Path:
    member = resolve_member(zf, names, relative_path)
    suffix = Path(member).suffix.lower() or '.jpg'
    key = hashlib.sha256((str(zf.filename) + '|' + member).encode('utf-8')).hexdigest()[:24]
    out = IMAGE_CACHE / f'{key}{suffix}'
    if not out.exists():
        with zf.open(member) as src, out.open('wb') as dst:
            dst.write(src.read())
    return out

zero_rows = read_jsonl(ZERO_QA)
one_rows = read_jsonl(ONE_QA)
struct_rows = read_jsonl(STRUCT_QA)
print(len(zero_rows), len(one_rows), len(struct_rows))


In [ ]:
# 6) 첫 시험 이미지를 눈으로 확인
from IPython.display import display

example = zero_rows[0]
print('asset:', example['asset_category'])
print('GT   :', example['answer'])
display(image_from_zip(UNSUP, UNSUP_NAMES, example['image']))


In [ ]:
# 7) Prompt + inference 함수

def zero_shot_messages(item):
    target_path = extract_image(UNSUP, UNSUP_NAMES, item['image'])
    prompt = (
        'Classify the TARGET power-line asset image as normal or anomalous. '
        'Reply with exactly one lowercase word: normal or anomalous. Do not explain.'
    )
    return [{
        'role': 'user',
        'content': [
            {'type': 'image', 'path': str(target_path)},
            {'type': 'text', 'text': prompt},
        ],
    }]


def one_shot_messages(item):
    ref_path = extract_image(UNSUP, UNSUP_NAMES, item['reference_image'])
    target_path = extract_image(UNSUP, UNSUP_NAMES, item['image'])
    return [{
        'role': 'user',
        'content': [
            {'type': 'text', 'text': 'REFERENCE IMAGE: a known normal example of the same asset category.'},
            {'type': 'image', 'path': str(ref_path)},
            {'type': 'text', 'text': 'TARGET IMAGE: classify this image.'},
            {'type': 'image', 'path': str(target_path)},
            {'type': 'text', 'text': 'Reply with exactly one lowercase word: normal or anomalous. Do not explain.'},
        ],
    }]


def structured_messages(item):
    target_path = extract_image(SUP, SUP_NAMES, item['image'])
    classes = ', '.join(item['valid_defect_classes'])
    prompt = (
        'Inspect the TARGET power-line asset image. Predict both anomaly status and defect class. '
        'The anomaly value must be either normal or anomalous. '
        f'The defect_class must be exactly one of: {classes}. '
        'Return exactly one JSON object with keys anomaly and defect_class, with no explanation. '
        'Example format: {"anomaly":"anomalous","defect_class":"rust"}'
    )
    return [{
        'role': 'user',
        'content': [
            {'type': 'image', 'path': str(target_path)},
            {'type': 'text', 'text': prompt},
        ],
    }]


def generate_text(messages, max_new_tokens=64):
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors='pt',
    )
    inputs.pop('token_type_ids', None)
    inputs = inputs.to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # benchmark reproducibility
        )
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    return processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


In [ ]:
# 8) 실제 1개 inference 먼저 확인
sample_output = generate_text(zero_shot_messages(zero_rows[0]), max_new_tokens=16)
print('GT    :', zero_rows[0]['answer'])
print('Model :', sample_output)


## 20-sample smoke test

기본값은 각 regime **20개씩**입니다.

- Zero-shot: 20
- One-shot: 20
- Structured: 20

즉 총 60개 prediction을 만듭니다. 비용/시간을 먼저 보고 싶으면 `LIMIT = 3`처럼 줄여도 됩니다.


In [ ]:
# 9) 결과 저장 위치 + batch runner
from datetime import datetime
from tqdm.auto import tqdm

LIMIT = 20
RUN_NAME = f'qwen3vl_smoke_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
RUN_DIR = DRIVE_ROOT / 'lvlm_insplad_results' / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    'model_id': MODEL_ID,
    'gpu': torch.cuda.get_device_name(0),
    'vram_gb': round(vram_gb, 2),
    'dtype': str(MODEL_DTYPE),
    'limit_per_regime': LIMIT,
    'branch': BRANCH,
}
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')


def run_rows(rows, message_builder, out_path: Path, max_new_tokens=64):
    records = []
    for item in tqdm(rows[:LIMIT]):
        try:
            output = generate_text(message_builder(item), max_new_tokens=max_new_tokens)
            error = None
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            raise RuntimeError(
                'CUDA OOM. MODEL_ID_OVERRIDE를 Qwen/Qwen3-VL-4B-Instruct로 바꾸고 '
                '모델 로드 셀부터 다시 실행하세요.'
            ) from exc
        except Exception as exc:
            output = ''
            error = f'{type(exc).__name__}: {exc}'
        records.append({
            'id': item['id'],
            'output': output,
            'error': error,
            'model_id': MODEL_ID,
        })
        out_path.write_text(
            ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in records),
            encoding='utf-8',
        )
    return records

ZERO_PRED = RUN_DIR / 'qwen_zero_shot_20.jsonl'
ONE_PRED = RUN_DIR / 'qwen_one_shot_20.jsonl'
STRUCT_PRED = RUN_DIR / 'qwen_structured_20.jsonl'
print('RUN_DIR:', RUN_DIR)


In [ ]:
# 10) Zero-shot 20
zero_pred = run_rows(zero_rows, zero_shot_messages, ZERO_PRED, max_new_tokens=16)
print('saved:', ZERO_PRED)


In [ ]:
# 11) One-shot 20
one_pred = run_rows(one_rows, one_shot_messages, ONE_PRED, max_new_tokens=16)
print('saved:', ONE_PRED)


In [ ]:
# 12) Structured 20
struct_pred = run_rows(struct_rows, structured_messages, STRUCT_PRED, max_new_tokens=64)
print('saved:', STRUCT_PRED)


In [ ]:
# 13) 기존 evaluator로 smoke-test 점수 계산
RESULTS_DIR = RUN_DIR / 'metrics'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def evaluate(task, qa_path: Path, pred_path: Path, regime=None):
    cmd = [
        sys.executable, str(REPO_DIR / 'scripts/evaluate.py'),
        '--task', task,
        '--qa', str(qa_path),
        '--predictions', str(pred_path),
        '--out-dir', str(RESULTS_DIR),
    ]
    if regime:
        cmd += ['--regime', regime]
    subprocess.run(cmd, check=True)

evaluate('anomaly', ZERO_QA, ZERO_PRED, 'zero_shot')
evaluate('anomaly', ONE_QA, ONE_PRED, 'one_shot')
evaluate('explainability', STRUCT_QA, STRUCT_PRED)
print('metrics saved:', RESULTS_DIR)


In [ ]:
# 14) 결과 표 확인
import pandas as pd

for csv_path in sorted(RESULTS_DIR.glob('*.csv')):
    print('\n###', csv_path.name)
    display(pd.read_csv(csv_path))


In [ ]:
# 15) GT / raw output / parsed output을 한 표로 저장
from insplad_benchmark.parsing import parse_anomaly_output, parse_structured_output


def prediction_map(records):
    return {r['id']: r for r in records}

zp = prediction_map(zero_pred)
op = prediction_map(one_pred)
sp = prediction_map(struct_pred)
review = []

for regime, rows, preds in [
    ('zero_shot', zero_rows[:LIMIT], zp),
    ('one_shot', one_rows[:LIMIT], op),
]:
    for item in rows:
        rec = preds[item['id']]
        parsed = parse_anomaly_output(rec['output'])
        review.append({
            'regime': regime,
            'id': item['id'],
            'asset_category': item['asset_category'],
            'gt_anomaly': item['answer'],
            'gt_defect': '',
            'raw_output': rec['output'],
            'pred_anomaly': parsed.anomaly,
            'pred_defect': '',
            'valid': parsed.valid,
            'error': rec['error'],
        })

for item in struct_rows[:LIMIT]:
    rec = sp[item['id']]
    parsed = parse_structured_output(rec['output'], set(item['valid_defect_classes']))
    review.append({
        'regime': 'structured',
        'id': item['id'],
        'asset_category': item['asset_category'],
        'gt_anomaly': item['answer']['anomaly'],
        'gt_defect': item['answer']['defect_class'],
        'raw_output': rec['output'],
        'pred_anomaly': parsed.anomaly,
        'pred_defect': parsed.defect_class,
        'valid': parsed.valid,
        'error': rec['error'],
    })

review_df = pd.DataFrame(review)
review_path = RUN_DIR / 'smoke_review.csv'
review_df.to_csv(review_path, index=False)
display(review_df)
print('saved:', review_path)


In [ ]:
# 16) ChatGPT에 전달하기 쉽게 결과 폴더 전체를 ZIP으로 묶기
archive_base = RUN_DIR.parent / RUN_NAME
archive_path = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=RUN_DIR))
print('DONE')
print('결과 폴더:', RUN_DIR)
print('업로드용 ZIP:', archive_path)
print('\n이 ZIP을 ChatGPT에 올리면 zero/one/structured 실패 사례와 다음 개선 방향을 분석할 수 있습니다.')


## 다음 단계

이 smoke-test ZIP을 확인한 뒤에만 전체 inference로 확대합니다.

- Track A Zero-shot: 6,481
- Track A One-shot: 6,481
- Track B Structured: 6,417
- Aligned Core: 1,378 same-image 비교

첫 20개 단계에서 확인할 것:

- 이미지가 실제로 정상 로딩되는지
- zero-shot / one-shot prompt가 의도대로 작동하는지
- structured 출력이 JSON 형식을 잘 지키는지
- Invalid Output이 많은지
- 어떤 asset / defect에서 오답이 집중되는지
